In [82]:
import pandas as pd 
import numpy as np 


In [83]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder

In [84]:
df=pd.read_csv('/Users/prajwalsubedi/Desktop/Data science/Data Sets/covid_dataset.csv')

In [85]:
df.head()

,Age,Gender,Fever,Cough,City,Has_Covid
0,56,Male,102.4,Mild,Mumbai,No
1,19,Female,101.6,Strong,Mumbai,No
2,76,Female,102.0,Mild,Kolkata,No
3,65,Male,99.7,Mild,Bangalore,No
4,25,Female,102.1,Mild,Delhi,Yes


In [86]:
df.isnull().sum()

Age          0
Gender       0
Fever        0
Cough        0
City         0
Has_Covid    0
dtype: int64

In [87]:
x_train, x_test, y_train, y_test= train_test_split(df.drop(columns=['Has_Covid']),df['Has_Covid'],test_size=0.2,random_state=40)


In [88]:
x_train.head()

,Age,Gender,Fever,Cough,City
970,6,Female,101.0,Strong,Bangalore
137,75,Male,101.4,Strong,Mumbai
82,45,Female,98.7,Mild,Mumbai
188,48,Male,98.4,Strong,Delhi
358,46,Male,100.5,Mild,Delhi


In [89]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler

preprocessor = ColumnTransformer([
    
    # One-hot encode City (column 4)
    ('ohe_city', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [4]),

    # Ordinal encode Cough (column 3)
    ('oe_cough', OrdinalEncoder(categories=[['Mild', 'Strong']]), [3]),

    # Ordinal encode Gender (column 1)
    ('oe_gender', OrdinalEncoder(categories=[['Male', 'Female']]), [1]),

    # Scale numeric columns
    ('scale', MinMaxScaler(), [0, 2])  # adjust based on your dataset

], remainder='passthrough')

In [90]:
# one hot encoding 
# trf1 = ColumnTransformer([
#     ('ohe_city',OneHotEncoder(sparse_output=False, handle_unknown='ignore'),[4])
# ],remainder='passthrough')

In [91]:
# trf2 = ColumnTransformer(transformers=[
#     ('oe_cough_gender',OrdinalEncoder(categories=[['Mild','Strong'],['Male','Female']]),[3,1]),
# ],remainder='passthrough')

In [92]:
# trf3 = ColumnTransformer(transformers=[
#     ('le_Has_covid',LabelEncoder(),[5])
# ])

In [93]:
# trf4 = ColumnTransformer(transformers=[
#     ('scale',MinMaxScaler(),slice(0,10))
# ])

In [94]:
trf5= SelectKBest(score_func=chi2,k=8)

In [95]:
trf6= DecisionTreeClassifier()

# Creating Pipeline 

In [96]:
pipe = Pipeline([
    ('preprocessor',preprocessor),
    ('trf5',trf5),
    ('trf6',trf6)
])

In [97]:
# pipe = Pipeline([
#     ('trf1',trf1),
#     ('trf2',trf2),
#     ('trf3',trf3),
#     ('trf4',trf4),
#     ('trf5',trf5),
#     ('trf6',trf6)
# ])

# Pipeline Vs make_pipeline

### pipeline require naming of steps where make_pipeline doesnt.

In [98]:
# pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5,trf6)

In [99]:
pipe.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('trf5', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe_city', ...), ('oe_cough', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the

In [100]:
pipe.named_steps

{'preprocessor': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_city',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [4]),
                                 ('oe_cough',
                                  OrdinalEncoder(categories=[['Mild',
                                                              'Strong']]),
                                  [3]),
                                 ('oe_gender',
                                  OrdinalEncoder(categories=[['Male',
                                                              'Female']]),
                                  [1]),
                                 ('scale', MinMaxScaler(), [0, 2])]),
 'trf5': SelectKBest(k=8, score_func=<function chi2 at 0x17af98f40>),
 'trf6': DecisionTreeClassifier()}

In [101]:
y_pred =pipe.predict(x_test)

In [102]:
y_pred

array(['No', 'Yes', 'Yes', 'Yes', 'No', 'No', 'No', 'Yes', 'Yes', 'Yes',
       'No', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'Yes',
       'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'Yes', 'Yes', 'No',
       'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'No',
       'Yes', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No',
       'Yes', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No', 'Yes', 'Yes',
       'No', 'No', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'No',
       'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No',
       'No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'No', 'Yes',
       'Yes', 'Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes',
       'Yes', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'Yes',
       'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes',
       'Yes', 'No', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'No', '

In [103]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.45

# Cross validation using Pipeline

In [104]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe,x_train,y_train,cv=5, scoring='accuracy').mean()

np.float64(0.48374999999999996)

In [105]:
# GridSearch using Pipeline

In [106]:
params ={
    'trf6__max_depth':[1,2,3,4,5,None]
}

In [107]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(x_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'trf6__max_depth': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and cand

In [108]:
grid.best_score_

np.float64(0.5)

In [109]:
grid.best_params_

{'trf6__max_depth': 1}

# Exporting the pipeline


In [110]:
import pickle
pickle.dump(pipe, open('pipe.pkl','wb'))

In [111]:
x_train

,Age,Gender,Fever,Cough,City
970,6,Female,101.0,Strong,Bangalore
137,75,Male,101.4,Strong,Mumbai
82,45,Female,98.7,Mild,Mumbai
188,48,Male,98.4,Strong,Delhi
358,46,Male,100.5,Mild,Delhi
...,...,...,...,...,...
440,16,Female,100.4,Strong,Kolkata
165,48,Female,100.8,Strong,Kolkata
7,28,Male,100.7,Mild,Mumbai
219,43,Male,100.4,Mild,Kolkata
